# Interface de comparaison WikiText / PubMed

Cette interface compare deux versions de NexusLM à partir du même prompt : un modèle entraîné sur WikiText et un modèle entraîné sur PubMed.

Elle permet de modifier la température, le nombre de mots générés et le top-k, puis d'observer les textes produits par chaque modèle. L'interface affiche aussi les tokens du prompt, leurs identifiants et les prochains tokens les plus probables.

L'objectif est de montrer l'influence du corpus d'entraînement. WikiText produit plutôt un vocabulaire généraliste, tandis que PubMed tend davantage vers un vocabulaire scientifique ou biomédical. Les générations restent limitées car NexusLM est un petit modèle pédagogique, mais cette comparaison permet de visualiser concrètement l'effet des données sur le comportement du modèle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import tensorflow as tf
import numpy as np
import pickle
import re


In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout, LayerNormalization, Embedding
import pickle
import numpy as np

class PositionalEmbedding(Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = Embedding(vocab_size, embed_dim)
        self.position_embeddings = Embedding(sequence_length, embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        positions = tf.range(tf.shape(inputs)[-1])
        return self.token_embeddings(inputs) + self.position_embeddings(positions)

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim
        })
        return config


@tf.keras.utils.register_keras_serializable()
class MultiHeadAttentionLayer(Layer):
    def __init__(self, embed_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.d_head = embed_dim // num_heads
        self.embed_dim = embed_dim

        self.Wq = Dense(embed_dim, use_bias=False)
        self.Wk = Dense(embed_dim, use_bias=False)
        self.Wv = Dense(embed_dim, use_bias=False)
        self.Wo = Dense(embed_dim, use_bias=False)

    def causal_mask(self, batch_size, num_heads, seq_len):
        mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        mask = tf.reshape(mask, (1, 1, seq_len, seq_len))
        return tf.tile(mask, [batch_size, num_heads, 1, 1])

    def split_heads(self, x, B):
        x = tf.reshape(x, (B, -1, self.num_heads, self.d_head))
        return tf.transpose(x, [0, 2, 1, 3])

    def call(self, x):
        B = tf.shape(x)[0]
        seq_len = tf.shape(x)[1]

        Q = self.split_heads(self.Wq(x), B)
        K = self.split_heads(self.Wk(x), B)
        V = self.split_heads(self.Wv(x), B)

        scores = tf.matmul(Q, K, transpose_b=True)
        scores = scores / tf.sqrt(tf.cast(self.d_head, tf.float32))

        mask = self.causal_mask(B, self.num_heads, seq_len)
        scores = scores * mask + (1.0 - mask) * -1e9

        A = tf.nn.softmax(scores, axis=-1)

        context = tf.matmul(A, V)
        context = tf.transpose(context, [0, 2, 1, 3])
        context = tf.reshape(context, (B, -1, self.embed_dim))

        return self.Wo(context)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads
        })
        return config


@tf.keras.utils.register_keras_serializable()
class TransformerDecoderBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate

        self.att = MultiHeadAttentionLayer(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.drop1 = Dropout(dropout_rate)
        self.drop2 = Dropout(dropout_rate)

    def call(self, x, training=False):
        attn_out = self.att(x)
        attn_out = self.drop1(attn_out, training=training)
        x = self.norm1(x + attn_out)

        ffn_out = self.ffn(x)
        ffn_out = self.drop2(ffn_out, training=training)

        return self.norm2(x + ffn_out)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout_rate": self.dropout_rate
        })
        return config


@tf.keras.utils.register_keras_serializable()
class NexusLM(tf.keras.Model):
    def __init__(self, sequence_length, vocab_size, embed_dim,
                 num_heads, ff_dim, num_layers, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.num_layers = num_layers

        self.pos_embedding = PositionalEmbedding(sequence_length, vocab_size, embed_dim)
        self.decoder_blocks = [
            TransformerDecoderBlock(embed_dim, num_heads, ff_dim)
            for _ in range(num_layers)
        ]
        self.layernorm = LayerNormalization(epsilon=1e-6)
        self.dropout = Dropout(0.1)
        self.output_layer = Dense(vocab_size, activation="softmax")

    def call(self, inputs, training=False):
        x = self.pos_embedding(inputs)
        x._keras_mask = None

        for block in self.decoder_blocks:
            x = block(x, training=training)

        x = self.layernorm(x)
        x = self.dropout(x, training=training)

        return self.output_layer(x)

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "num_layers": self.num_layers
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)


@tf.keras.utils.register_keras_serializable()
class Nexus(NexusLM):
    pass


In [ ]:
custom_objects = {
    "PositionalEmbedding": PositionalEmbedding,
    "MultiHeadAttentionLayer": MultiHeadAttentionLayer,
    "TransformerDecoderBlock": TransformerDecoderBlock,
    "NexusLM": NexusLM,
}

model_pubmed = tf.keras.models.load_model("model_pubmed.keras", compile=False, custom_objects=custom_objects)
model_wiki = tf.keras.models.load_model("model_wiki.keras", compile=False, custom_objects=custom_objects)



In [ ]:
def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z']+", text)


In [ ]:
CTX = 32

def generer(model, seed_text, w2i, i2w, sequence_length=CTX,
            num_words=40, temperature=0.8, top_k=10):

    mots = tokenize(seed_text)
    encoded = [w2i.get(mot, 1) for mot in mots]

    if len(encoded) == 0:
        return "Aucun mot du prompt n'est connu par le vocabulaire."

    generated = encoded[:]

    for _ in range(num_words):
        seq = generated[-sequence_length:]
        seq = [0] * (sequence_length - len(seq)) + seq
        x = np.array([seq])

        pred = model.predict(x, verbose=0)

        if pred.ndim == 3:
            probs = pred[0, -1]
        else:
            probs = pred[0]

        probs = np.array(probs, dtype=np.float64)

        probs[0] = 0
        probs[1] = 0

        probs = np.log(probs + 1e-8) / temperature
        probs = np.exp(probs - np.max(probs))
        probs = probs / np.sum(probs)

        top_indices = np.argsort(probs)[-top_k:]
        top_probs = probs[top_indices]
        top_probs = top_probs / np.sum(top_probs)

        next_id = int(np.random.choice(top_indices, p=top_probs))
        generated.append(next_id)

    return " ".join(i2w.get(i, "<UNK>") for i in generated)


In [ ]:
!pip install -q ipywidgets


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prompt_box = widgets.Text(
    value="the patient received",
    description="Prompt:",
    layout=widgets.Layout(width="80%")
)

temperature = widgets.FloatSlider(
    value=0.8,
    min=0.5,
    max=1.5,
    step=0.1,
    description="Temp:"
)

button = widgets.Button(description="Comparer")
output = widgets.Output()

def comparer(b):
    with output:
        clear_output()

        prompt = prompt_box.value
        temp = temperature.value

        print("Prompt :", prompt)
        print("Température :", temp)

        print("\n--- NexusLM WikiText ---")
        print(generer(model_wiki, prompt, w2i_wiki, i2w_wiki, CTX, temperature=temp))

        print("\n--- NexusLM PubMed ---")
        print(generer(model_pubmed, prompt, w2i_pubmed, i2w_pubmed, CTX, temperature=temp))

button.on_click(comparer)

display(prompt_box, temperature, button, output)


In [ ]:
import pickle

with open("vocab_wiki.pkl", "rb") as f:
    w2i_wiki, i2w_wiki = pickle.load(f)

with open("vocab_pubmed.pkl", "rb") as f:
    w2i_pubmed, i2w_pubmed = pickle.load(f)

CTX = 32

print("Vocab WikiText :", len(w2i_wiki))
print("Vocab PubMed :", len(w2i_pubmed))


In [ ]:
prompts = [
    "the patient received",
    "the treatment of",
    "clinical study showed",
    "in this study",
    "the result of",
    "patients with disease"
]

for prompt in prompts:
    print("\n==============================")
    print("Prompt :", prompt)

    print("\nWikiText :")
    print(generer(model_wiki, prompt, w2i_wiki, i2w_wiki, CTX, temperature=0.8))

    print("\nPubMed :")
    print(generer(model_pubmed, prompt, w2i_pubmed, i2w_pubmed, CTX, temperature=0.8))


In [ ]:
!pip install -q gradio

import gradio as gr
import pandas as pd
import numpy as np
import re

CTX = 32
NUM_HEADS = 4
NUM_LAYERS = 2

theme_nexus = gr.themes.Soft(
    primary_hue="violet",
    neutral_hue="slate"
)

css_nexus = """
.gradio-container {
    max-width: 1500px !important;
    margin: auto !important;
}

body {
    background: #0b1020 !important;
}

#header {
    background: linear-gradient(135deg, #111827, #4c1d95);
    color: white;
    padding: 18px 28px;
    border-radius: 16px;
    margin-bottom: 14px;
}

#header h1 {
    margin: 0;
    font-size: 32px;
}

#header p {
    margin: 6px 0 0 0;
    font-size: 16px;
    opacity: 0.9;
}

textarea {
    font-size: 14px !important;
}

button {
    font-weight: bold !important;
}
"""

def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z']+", text)

def tableau_tokens(prompt, w2i, nom_modele):
    mots = tokenize(prompt)
    lignes = []

    for mot in mots:
        token_id = w2i.get(mot, 1)
        statut = "inconnu" if mot not in w2i else "connu"
        lignes.append([nom_modele, mot, token_id, statut])

    return pd.DataFrame(lignes, columns=["modèle", "token", "id", "statut"])

def top_predictions(model, prompt, w2i, i2w, temperature=0.8, top_k=5):
    temperature = float(temperature)
    top_k = int(top_k)

    mots = tokenize(prompt)
    encoded = [w2i.get(mot, 1) for mot in mots]

    seq = encoded[-CTX:]
    seq = [0] * (CTX - len(seq)) + seq
    x = np.array([seq])

    prediction = model.predict(x, verbose=0)

    if prediction.ndim == 3:
        prediction = prediction[0, -1]
    else:
        prediction = prediction[0]

    prediction = np.log(prediction + 1e-8) / temperature
    prediction = np.exp(prediction) / np.sum(np.exp(prediction))

    indices = np.argsort(prediction)[-top_k:][::-1]

    lignes = []
    for idx in indices:
        token = i2w.get(int(idx), "?")
        proba = float(prediction[idx])
        lignes.append([token, int(idx), round(proba, 4)])

    return pd.DataFrame(lignes, columns=["token proposé", "id", "probabilité"])

def resume_calcul(prompt, nb_mots):
    mots = tokenize(prompt)
    nb_tokens_prompt = len(mots)
    scores_par_token = NUM_LAYERS * NUM_HEADS * (CTX ** 2)
    total = scores_par_token * int(nb_mots) * 2

    return f"""
### Résumé du calcul

- Tokens dans le prompt : **{nb_tokens_prompt}**
- Taille du contexte : **{CTX}**
- Nombre de couches : **{NUM_LAYERS}**
- Nombre de têtes : **{NUM_HEADS}**
- Scores d'attention pour prédire un token : **{scores_par_token}**
- Approximation pour comparer les deux modèles : **{total}**
"""

def demo_nexus_compact(prompt, temperature, nb_mots, top_k, progress=gr.Progress()):
    temperature = float(temperature)
    nb_mots = int(nb_mots)
    top_k = int(top_k)

    progress(0.15, desc="Tokenisation")

    tokens_wiki = tableau_tokens(prompt, w2i_wiki, "WikiText")
    tokens_pubmed = tableau_tokens(prompt, w2i_pubmed, "PubMed")
    tokens = pd.concat([tokens_wiki, tokens_pubmed], ignore_index=True)

    progress(0.35, desc="Génération WikiText")

    texte_wiki = generer(
        model_wiki,
        prompt,
        w2i_wiki,
        i2w_wiki,
        CTX,
        num_words=nb_mots,
        temperature=temperature,
        top_k=top_k
    )

    progress(0.60, desc="Génération PubMed")

    texte_pubmed = generer(
        model_pubmed,
        prompt,
        w2i_pubmed,
        i2w_pubmed,
        CTX,
        num_words=nb_mots,
        temperature=temperature,
        top_k=top_k
    )

    progress(0.80, desc="Prédictions suivantes")

    top_wiki = top_predictions(
        model_wiki,
        prompt,
        w2i_wiki,
        i2w_wiki,
        temperature=temperature,
        top_k=top_k
    )

    top_pubmed = top_predictions(
        model_pubmed,
        prompt,
        w2i_pubmed,
        i2w_pubmed,
        temperature=temperature,
        top_k=top_k
    )

    stats = resume_calcul(prompt, nb_mots)

    progress(1.0, desc="Terminé")

    return tokens, stats, texte_wiki, top_wiki, texte_pubmed, top_pubmed

with gr.Blocks(theme=theme_nexus, css=css_nexus, title="NexusLM") as demo:
    gr.HTML("""
    <div id="header">
        <h1>NexusLM</h1>
        <p>Démo comparaison WikiText / PubMed</p>
    </div>
    """)

    with gr.Row():
        prompt = gr.Textbox(
            label="Prompt",
            value="the patient received",
            lines=1,
            scale=4
        )

        temperature = gr.Slider(
            0.5,
            1.5,
            value=0.8,
            step=0.1,
            label="Température",
            scale=1
        )

        nb_mots = gr.Slider(
            10,
            50,
            value=25,
            step=5,
            label="Mots",
            scale=1
        )

        top_k = gr.Slider(
            3,
            10,
            value=5,
            step=1,
            label="Top-k",
            scale=1
        )

        bouton = gr.Button(
            "Générer",
            variant="primary",
            scale=1
        )

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            gr.Markdown("### Prompt")
            table_tokens = gr.Dataframe(
                label="Tokens et identifiants"
            )
            stats = gr.Markdown()

        with gr.Column(scale=1):
            gr.Markdown("### Nexus WikiText")
            sortie_wiki = gr.Textbox(
                label="Texte généré",
                lines=7
            )
            table_top_wiki = gr.Dataframe(
                label="Prochains tokens probables"
            )

        with gr.Column(scale=1):
            gr.Markdown("### Nexus PubMed")
            sortie_pubmed = gr.Textbox(
                label="Texte généré",
                lines=7
            )
            table_top_pubmed = gr.Dataframe(
                label="Prochains tokens probables"
            )

    bouton.click(
        fn=demo_nexus_compact,
        inputs=[prompt, temperature, nb_mots, top_k],
        outputs=[
            table_tokens,
            stats,
            sortie_wiki,
            table_top_wiki,
            sortie_pubmed,
            table_top_pubmed
        ],
        show_progress="full"
    )

demo.launch()


In [ ]:
print("model_wiki :", type(model_wiki))
print("model_pubmed :", type(model_pubmed))

print("vocab wiki :", len(w2i_wiki))
print("vocab pubmed :", len(w2i_pubmed))

print("test génération wiki :")
print(generer(model_wiki, "the patient received", w2i_wiki, i2w_wiki, CTX, num_words=10, temperature=0.8, top_k=5))

print("test génération pubmed :")
print(generer(model_pubmed, "the patient received", w2i_pubmed, i2w_pubmed, CTX, num_words=10, temperature=0.8, top_k=5))
